In [1]:
# Classical NLP Pipeline for Sentiment Classification

## 1. Introduction
## 2. Data Loading
## 3. Preprocessing
## 4. Feature Extraction
## 5. Logistic Regression
## 6. Neural Network
## 7. Evaluation
## 8. Discussion

In [2]:
import numpy as np
import pandas as pd
import re

from datasets import load_dataset

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

/Users/chitranshgupta/venv/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/Users/chitranshgupta/venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


In [3]:
dataset = load_dataset("imdb")

X_train = dataset['train']['text']
y_train = dataset['train']['label']

X_test = dataset['test']['text']
y_test = dataset['test']['label']

In [4]:
def preprocess(text):
    text = text.lower()
    text = re.sub(r"<.*?>", "", text)        # remove HTML tags
    text = re.sub(r"[^a-z\s]", "", text)    # keep only letters
    text = re.sub(r"\s+", " ", text).strip()
    return text

In [5]:
X_train = [preprocess(x) for x in X_train]
X_test = [preprocess(x) for x in X_test]

In [6]:
tfidf_uni = TfidfVectorizer(max_features=5000, ngram_range=(1,1))

X_train_uni = tfidf_uni.fit_transform(X_train)
X_test_uni = tfidf_uni.transform(X_test)

In [7]:
tfidf_bi = TfidfVectorizer(max_features=5000, ngram_range=(1,2))

X_train_bi = tfidf_bi.fit_transform(X_train)
X_test_bi = tfidf_bi.transform(X_test)

In [8]:
lr_uni = LogisticRegression(max_iter=1000)
lr_uni.fit(X_train_uni, y_train)

y_pred_uni = lr_uni.predict(X_test_uni)

In [9]:
lr_bi = LogisticRegression(max_iter=1000)
lr_bi.fit(X_train_bi, y_train)

y_pred_bi = lr_bi.predict(X_test_bi)

In [10]:
print("Logistic Regression (Unigram)")
print("Accuracy:", accuracy_score(y_test, y_pred_uni))
print(classification_report(y_test, y_pred_uni))

Logistic Regression (Unigram)
Accuracy: 0.88168
              precision    recall  f1-score   support

           0       0.88      0.88      0.88     12500
           1       0.88      0.89      0.88     12500

    accuracy                           0.88     25000
   macro avg       0.88      0.88      0.88     25000
weighted avg       0.88      0.88      0.88     25000



In [11]:
print("Logistic Regression (Bigram)")
print("Accuracy:", accuracy_score(y_test, y_pred_bi))
print(classification_report(y_test, y_pred_bi))

Logistic Regression (Bigram)
Accuracy: 0.88552
              precision    recall  f1-score   support

           0       0.89      0.88      0.88     12500
           1       0.88      0.89      0.89     12500

    accuracy                           0.89     25000
   macro avg       0.89      0.89      0.89     25000
weighted avg       0.89      0.89      0.89     25000



In [12]:
X_train_tensor = torch.tensor(X_train_uni.toarray(), dtype=torch.float32)
X_test_tensor = torch.tensor(X_test_uni.toarray(), dtype=torch.float32)

y_train_tensor = torch.tensor(y_train)
y_test_tensor = torch.tensor(y_test)

In [13]:
class SimpleNN(nn.Module):
    def __init__(self, input_dim):
        super(SimpleNN, self).__init__()
        self.fc1 = nn.Linear(input_dim, 128)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(128, 2)

    def forward(self, x):
        x = self.relu(self.fc1(x))
        x = self.fc2(x)
        return x

In [14]:
model = SimpleNN(X_train_tensor.shape[1])
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

for epoch in range(5):
    outputs = model(X_train_tensor)
    loss = criterion(outputs, y_train_tensor)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    print(f"Epoch {epoch+1}, Loss: {loss.item():.4f}")

Epoch 1, Loss: 0.6936
Epoch 2, Loss: 0.6908
Epoch 3, Loss: 0.6874
Epoch 4, Loss: 0.6832
Epoch 5, Loss: 0.6783


In [15]:
with torch.no_grad():
    outputs = model(X_test_tensor)
    _, predicted = torch.max(outputs, 1)

print("Neural Network Results")
print("Accuracy:", accuracy_score(y_test, predicted))
print(classification_report(y_test, predicted))

Neural Network Results
Accuracy: 0.71956
              precision    recall  f1-score   support

           0       0.65      0.96      0.77     12500
           1       0.92      0.48      0.63     12500

    accuracy                           0.72     25000
   macro avg       0.78      0.72      0.70     25000
weighted avg       0.78      0.72      0.70     25000



In [16]:
results = pd.DataFrame({
    "Model": ["LogReg", "LogReg", "Neural Net"],
    "Features": ["TF-IDF (uni)", "TF-IDF (bi)", "TF-IDF (uni)"],
    "Accuracy": [
        accuracy_score(y_test, y_pred_uni),
        accuracy_score(y_test, y_pred_bi),
        accuracy_score(y_test, predicted)
    ]
})

print(results)

        Model      Features  Accuracy
0      LogReg  TF-IDF (uni)   0.88168
1      LogReg   TF-IDF (bi)   0.88552
2  Neural Net  TF-IDF (uni)   0.71956
